# 第9回 顔画像を、少ない数で持つ（解答例・教員用）


---


## 前回からの接続


前回は、2,000本の遺伝子を一緒に動くまとまりに分け、まとまりごとの平均で、たくさんの列を10本ほどの点数に縮めました。平均は、いちばん単純な縮め方でした。

今日は、たくさんの数でできたデータを、できるだけ情報を失わずに少ない数に縮める方法を学びます。題材は、顔の画像です。顔の画像1枚は、4,096個の数でできています。


---


## 今日の分析目標


あなたは、顔認証を作る開発チームの一員です。顔の画像1枚（4,096画素）を、誰の顔かを見分ける力を保ったまま、何個の数で持てばよいかに答える。

40人の顔の画像400枚を使います。データを、平均からのずれの誤差がいちばん小さくなる向きで近似する方法（主成分分析）を、2画素の小さな例から組み立て、4,096画素の顔に広げます。

もう1つのねらいは、表を行列として見て、縮める・戻すを行列の積で書けるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、建物の入館ゲートの顔認証を作る開発チームの一員です。前もって登録した人の顔の画像と、カメラに写った顔を比べて、誰の顔かを当てます。

顔の画像1枚は、4,096個の数です。登録した人が増えると、そのまま持つと記憶も重く、比べる計算も重くなります。1枚を、もっと少ない数で持てないでしょうか。ただし、少なくしすぎて、誰の顔かを見分けられなくなっては困ります。


### 手元のデータ

英国ケンブリッジのAT&T研究所が1992〜1994年に撮った顔の画像です（Olivettiの顔画像。scikit-learnが配布しています）。40人について、1人10枚ずつ、表情や眼鏡の有無、顔の向きを少しずつ変えて撮ってあります。

1枚は、縦64 × 横64＝4,096画素の白黒の画像です。値は画素の明るさで、0が黒、1が白です。1行が1枚で、先頭の列`person`が人の番号（0〜39）、残りの4,096列が画素の明るさです。


### 最後に出す答え

顔の画像1枚を何個の数で持てばよいかと、その数で持ったときに、誰の顔かを当てる力がどれだけ残るか（正解率）です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 画像を数の並び（行）として表し、`reshape`で画像に戻せる
- 平均からのずれを、1本の向きで近似し、誤差がいちばん小さい向きを探せる。それが散らばりのいちばん大きい向きでもあることを説明できる
- 主成分分析で向き（主成分）と得点を求め、向きを顔として描いて読める
- 得点＝ずれ`@`向きの転置、再構成＝得点`@`向き＋平均、を行列の積で書ける
- 累積寄与率と再構成誤差で、何本の主成分でどれだけ残るかを確かめられる
- 1-最近傍法で顔を見分け、主成分分析をパイプライン（pipeline）に入れて、交差検証（cross-validation）で見分ける力を比べられる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）

今日は、主成分分析の道具`PCA`と、いちばん近いものを探して当てる道具`KNeighborsClassifier`が加わります。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.model_selection import cross_val_score


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 練習用の道具：顔を並べて描く

- 役割：4,096個の数の並びを、何枚か横に並べて、顔の画像として描く
- 入力：顔の並びのリスト`faces`と、それぞれの見出し`titles`
- 出力：顔の画像を横に並べた図

今日のために用意した道具です。中身は読み飛ばしてかまいません。1枚の描き方は、1節で読みます。


In [ ]:
def show_faces(faces, titles):
    n = len(faces)
    plt.figure(figsize=(2.2 * n, 2.6))
    for k in range(n):
        plt.subplot(1, n, k + 1)
        plt.imshow(faces[k].reshape(64, 64), cmap='gray')
        plt.title(titles[k], fontsize=12)
        plt.axis('off')
    plt.show()


### お手本：顔の画像の表を読み込む

- 役割：顔の画像の表を読み込み、大きさと左上を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名
- 出力：表の大きさと、左上の3行・5列

表は大きいので圧縮してあります（`.csv.gz`。第8回と同じ）。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/olivetti_faces.csv.gz', index_col=0)
print(df.shape)
df.iloc[:3, :5]


400行 × 4,097列です。1行が1枚の画像で、先頭の列`person`が人の番号、`px0`から`px4095`までの4,096列が画素の明るさです。


### お手本：人の番号と画素に分け、訓練とテストに分ける

- 役割：人の番号を`person`、画素の明るさを`X`にし、訓練75%・テスト25%に分ける
- 入力：表`df`
- 出力：訓練とテストの大きさ

今日は、表を行列として計算するので、`.values`でnumpyの配列にします。`stratify=person`は、どの人も訓練とテストに同じ割合で入るように分ける指定です（第6回）。テストの100枚は、最後に1回だけ使います。


In [ ]:
person = df['person'].values
X = df.drop(columns='person').values
X_tr, X_te, p_tr, p_te = train_test_split(
    X, person, test_size=0.25, random_state=0, stratify=person)
print(X_tr.shape, X_te.shape)


訓練は300枚、テストは100枚です。どちらも、1枚が4,096個の数です。


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、② データ解析（縮める方法）を中心に、③ 結果の解釈（何個で持つか）まで進みます。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の7つの小さな問いに答えていきます。

1. 顔は数の表か？（1節）
2. 1本の向きで近似するには？（2節）
3. 顔の向きはどう見えるか？（3節）
4. 少ない数から顔を戻せるか？（4節）
5. 何本でどれだけ残るか？（5節）
6. 見分ける力は残るか？（6節）
7. 何個の数で持てばよいか？（7節）


---


## 1. 顔は数の表か？


表が読めました。1行の4,096個の数が、本当に顔なのかを確かめます。


### おさらい：並びの形を変える

`reshape`は、数の並びの形を変えます。6個の並びを2行×3列にすると、左から順に、1行目、2行目と詰めていきます。


In [ ]:
a = np.arange(6)                  # 0, 1, 2, 3, 4, 5
print(a.reshape(2, 3))


画像の4,096個も、左上から右へ、1行ずつ並べたものです。`reshape(64, 64)`で、64行×64列の画像の形に戻せます。


In [ ]:
# @title 図：画像の4×4の部分と、1行に並べた形
img = X_tr[0].reshape(64, 64)
patch = img[30:34, 30:34]
fig, axes = plt.subplots(1, 2, figsize=(11, 3), gridspec_kw={'width_ratios': [1, 3]})
axes[0].imshow(patch, cmap='gray', vmin=0, vmax=1)
for r in range(4):
    for c in range(4):
        v = patch[r, c]
        axes[0].text(c, r, f'{v:.2f}', ha='center', va='center',
                     color='white' if v < 0.5 else 'black', fontsize=10)
axes[0].axis('off'); axes[0].set_title('画像の4×4の部分')
row = patch.ravel()
axes[1].imshow(row.reshape(1, -1), cmap='gray', vmin=0, vmax=1, aspect='auto')
for c in range(16):
    axes[1].text(c, 0, f'{row[c]:.2f}', ha='center', va='center', rotation=90,
                 color='white' if row[c] < 0.5 else 'black', fontsize=9)
axes[1].set_yticks([]); axes[1].set_xticks([0, 4, 8, 12])
axes[1].set_xticklabels(['1行目', '2行目', '3行目', '4行目'])
axes[1].set_title('1行に並べた形')
plt.show()


### お手本：1行を顔に戻す

- 役割：訓練の1枚目の4,096個を64行×64列に戻し、画像として描く
- 入力：訓練の画素`X_tr`と人の番号`p_tr`
- 出力：顔の画像

`plt.imshow`は、数の表を画像として描きます。`cmap='gray'`で、小さい数を黒、大きい数を白にします。


In [ ]:
img = X_tr[0].reshape(64, 64)       # 4,096個を64行×64列に
plt.imshow(img, cmap='gray')
plt.title(f'人の番号 {p_tr[0]}')
plt.axis('off')
plt.show()


4,096個の数が、顔になりました。この人の番号は37です。


### 変えてみる：同じ人の写真を並べる

- 役割：訓練の1枚目と同じ人の写真を取り出し、枚数を数えて、4枚を並べる
- 入力：訓練の画素と人の番号
- 出力：枚数と、顔の並び

`X_tr[p_tr == 37]`は、人の番号が37の行だけを残します（第1回の条件で絞る形を、配列に使う形です）。並べて描くのは、0節で用意した`show_faces`です。


In [ ]:
same = X_tr[p_tr == 37]
print(len(same), '枚')
show_faces(same[:4], ['1枚目', '2枚目', '3枚目', '4枚目'])


番号37の人は、訓練に8枚あります。同じ人でも、表情や顔の向きが少しずつ違います。この違いがあっても、誰の顔かを見分けたいのです。


### お手本：平均の顔

- 役割：訓練の300枚で、画素ごとに明るさの平均を出し、顔として描く
- 入力：訓練の画素`X_tr`
- 出力：平均の顔

`X_tr.mean(axis=0)`は、縦の方向に（300枚をまたいで）平均し、画素ごとの平均を4,096個出します。pandasの`mean()`は列ごとが決まりでしたが、numpyでは`axis=0`で縦の方向を指定します。


In [ ]:
mean_face = X_tr.mean(axis=0)       # 画素ごとの平均（4,096個）
print(mean_face.shape)
plt.imshow(mean_face.reshape(64, 64), cmap='gray')
plt.axis('off')
plt.show()


4,096個の平均が、ぼんやりした顔になりました。300枚の顔に共通する形です。


### 記号で書く：平均からのずれ

$i$枚目の画像の4,096個の数の並びを、$\mathbf{x}_i$と書きます。平均の顔を$\bar{\mathbf{x}}$と書くと、平均からのずれは$\mathbf{x}_i - \bar{\mathbf{x}}$です。

$$\bar{\mathbf{x}} = \frac{1}{n}\sum_{i=1}^{n} \mathbf{x}_i$$

$n$は枚数（訓練なら300）です。どの顔も「平均の顔＋その顔のずれ」です。顔ごとの違いは、ずれのほうにあります。


### TODO①：平均からのずれを描く

訓練の5枚目`X_tr[4]`と、平均の顔`mean_face`と、そのずれ`X_tr[4] - mean_face`の3つを、`show_faces`で並べてください。見出しは「写真」「平均」「ずれ」とします。


In [ ]:
# TODO: 5枚目・平均の顔・ずれ を並べて描く

# 解答例①：5枚目・平均の顔・ずれを並べる
show_faces([X_tr[4], mean_face, X_tr[4] - mean_face],
           ['写真', '平均', 'ずれ'])


ずれの画像では、平均より明るい画素が白く、暗い画素が黒くなります。ずれの4,096個が、その顔らしさを表しています。今日は、このずれを少ない数で表す方法を考えます。


---


## 2. 1本の向きで近似するには？


顔は、平均の顔とずれでできていました。ずれは4,096個の数で、4,096次元の図は描けません。そこでまず、2画素だけで考えます。2画素なら、散布図に描けます。


### お手本：2画素を取り出す

- 役割：左右の目のあたりの2画素（20行目の20列目と44列目）を取り出し、2つの明るさの相関係数を出す
- 入力：訓練の画素`X_tr`
- 出力：取り出した表の大きさと、相関係数

20行目の20列目は、左上から数えて$20 \times 64 + 20 = 1300$番目の数です。44列目は1324番目です。`X_tr[:, [1300, 1324]]`は、すべての行の、その2列を取り出します。`np.corrcoef`は相関係数の表を返すので、`[0, 1]`で2つの間の相関係数を取ります。


In [ ]:
two = X_tr[:, [1300, 1324]]        # 20行目の20列目と44列目
print(two.shape)
print(np.corrcoef(two[:, 0], two[:, 1])[0, 1].round(3))


300枚 × 2画素の表です。相関係数は0.817です。左の目のあたりが明るい写真は、右の目のあたりも明るい傾向があります。


In [ ]:
# @title 図：2画素の位置と、300枚の散布図
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].imshow(mean_face.reshape(64, 64), cmap='gray')
axes[0].plot([20, 44], [20, 20], 'o', color='#e63946', ms=9)
axes[0].axis('off'); axes[0].set_title('2画素の位置')
axes[1].scatter(two[:, 0], two[:, 1], s=12, alpha=0.6)
axes[1].set_xlabel('左の画素の明るさ'); axes[1].set_ylabel('右の画素の明るさ')
axes[1].set_aspect('equal')
plt.show()


### 変えてみる：平均からのずれにする

- 役割：2画素それぞれの平均を引き、平均からのずれの表`xc`を作る
- 入力：2画素の表`two`
- 出力：ずれの平均（ほぼ0になる）

1節と同じく、`mean(axis=0)`で列ごとの平均を出し、各行から引きます。


In [ ]:
xc = two - two.mean(axis=0)        # 平均からのずれ
print(xc.mean(axis=0).round(3))


ずれの平均は、どちらも0です。点の集まりの中心が、原点に来ました。


### 1本の向きで近似する

ずれの点は、右上がりの細長い形に集まっています。そこで、原点を通る1本の直線を引き、各点を直線上の点で近似することを考えます。

直線の向きを、長さ1の矢印$\mathbf{w} = (\cos\theta, \sin\theta)$で表します。$\theta$は、横軸から測った角度です。点$\mathbf{x}_i$（ここでは、ずれ）から直線に垂直に下ろした足が、近似の点です。

- 得点：$t_i = \mathbf{x}_i \mathbf{w}^\top$（向きに沿って、どれだけ進んだ所か）
- 近似：$t_i \mathbf{w}$（直線上の点）
- 誤差：$\sum_{i} \| \mathbf{x}_i - t_i \mathbf{w} \|^2$（点と近似の点の距離の二乗を、全員で足す）

点を直線に垂直に下ろすことを、射影（projection）と呼びます。誤差がいちばん小さくなる向きを探します。


In [ ]:
# @title 図：30度の向きに射影する
th = np.deg2rad(30)
w = np.array([np.cos(th), np.sin(th)])
t = xc @ w
plt.figure(figsize=(6, 5.5))
plt.scatter(xc[:, 0], xc[:, 1], s=12, alpha=0.4)
L = 0.45
plt.plot([-L * w[0], L * w[0]], [-L * w[1], L * w[1]], color='#e63946', lw=2)
for k in range(0, 300, 15):
    plt.plot([xc[k, 0], t[k] * w[0]], [xc[k, 1], t[k] * w[1]], color='#888', lw=1)
plt.scatter(t[::15] * w[0], t[::15] * w[1], s=20, color='#e63946', zorder=3)
plt.xlabel('左の画素のずれ'); plt.ylabel('右の画素のずれ')
plt.gca().set_aspect('equal')
plt.show()


### おさらい：行列の積と転置

第2回で、正規方程式を`@`と`.T`で書きました。`@`は行列の積、`.T`は転置（行と列の入れ替え）です。1行2列の行列を転置すると、2行1列になります。


In [ ]:
A = np.array([[1, 2]])             # 1行2列
print(A.T)                           # 2行1列
print(np.array([[3, 4]]) @ A.T)      # (1行2列) @ (2行1列) → 1行1列


$3 \times 1 + 4 \times 2 = 11$です。1行2列と2行1列の積は、2つの数を掛けて足した、1つの数になります。


### お手本：30度の向きで近似し、誤差を出す

- 役割：30度の向き$\mathbf{w}$を1行2列の行列`W`にし、得点・近似・誤差を計算する
- 入力：ずれの表`xc`
- 出力：得点と近似の大きさ、誤差

`np.deg2rad(30)`は、30度を、計算で使う角度の単位（ラジアン）に直します。`np.cos`・`np.sin`は、余弦と正弦です。得点は`xc @ W.T`で、300枚それぞれについて、2つの数を掛けて足します。近似は`t @ W`で、得点に向きを掛けます。


In [ ]:
theta = np.deg2rad(30)
W = np.array([[np.cos(theta), np.sin(theta)]])   # 向き：1行2列
t = xc @ W.T                       # 得点：300行1列
approx = t @ W                     # 近似：300行2列
print(t.shape, approx.shape)
print(round(((xc - approx) ** 2).sum(), 3))      # 誤差


30度の向きの誤差は、1.075です。


### 変えてみる：向きを0〜179度で回す

- 役割：角度を1度ずつ変えて、それぞれの誤差を出し、誤差がいちばん小さい角度を探す
- 入力：ずれの表`xc`
- 出力：いちばん小さい誤差の角度と、その誤差

お手本と同じ計算を`for`で180回くり返します。`np.argmin`は、いちばん小さい値の位置を返します（第7回）。角度は0度から1度ずつなので、位置がそのまま角度です。


In [ ]:
errs = []
for deg in range(180):
    theta = np.deg2rad(deg)
    W = np.array([[np.cos(theta), np.sin(theta)]])
    t = xc @ W.T
    errs.append(((xc - t @ W) ** 2).sum())
best = np.argmin(errs)
print(best, '度', round(errs[best], 3))


In [ ]:
plt.plot(range(180), errs)
plt.axvline(best, color='gray', ls='--')
plt.xlabel('向きの角度（度）')
plt.ylabel('誤差（距離の二乗の和）')
plt.show()


誤差は、45度でいちばん小さく、0.666です。点の集まりの細長い方向に、直線を合わせた形です。


### 名前：第1主成分

誤差がいちばん小さくなる向きを、第1主成分（principal component）と呼びます。点を第1主成分に射影した位置$t_i$を、主成分得点と呼びます。このように、平均からのずれを少ない本数の向きで近似する方法を、主成分分析（principal component analysis, PCA）と呼びます。

scikit-learnの`PCA`で、同じ向きが求まるかを確かめます。


### お手本：PCAで向きを求める

- 役割：2画素の表に主成分分析をかけ、第1主成分の向きを出す。45度の向きと比べる
- 入力：2画素の表`two`
- 出力：`PCA`の向きと、45度の向き

`PCA(n_components=1)`は、向きを1本だけ求める指定です。平均を引く計算は、`PCA`が中でします。求めた向きは`components_`に入ります。


In [ ]:
pca2 = PCA(n_components=1).fit(two)
print(pca2.components_.round(3))
theta = np.deg2rad(best)
print(np.round([np.cos(theta), np.sin(theta)], 3))


`PCA`の向きと、45度の向きは、ほぼ同じです（小さな違いは、1度ずつしか回していないためです）。符号が両方とも逆になることもありますが、矢印が逆向きなだけで、同じ直線です。


### 誤差が最小なら、得点の散らばりが最大

点$\mathbf{x}_i$と、近似の点$t_i \mathbf{w}$と、原点は、直角三角形を作ります。三平方の定理から、

$$\|\mathbf{x}_i\|^2 = t_i^2 + \|\mathbf{x}_i - t_i \mathbf{w}\|^2$$

全員で足すと、左辺の「ずれの大きさの合計」は、向きによらず一定です。だから、誤差（右辺の2つ目）が最小になる向きは、得点の二乗和（右辺の1つ目）が最大になる向きです。得点の平均は0なので、得点の二乗和が最大とは、得点の散らばり（分散）が最大ということです。


- 役割：45度の向きで、ずれの大きさの合計、得点の二乗和、誤差を出し、足し算が合うかを確かめる
- 入力：ずれの表`xc`、角度`best`
- 出力：3つの数


In [ ]:
theta = np.deg2rad(best)
W = np.array([[np.cos(theta), np.sin(theta)]])
t = xc @ W.T
print(round((xc ** 2).sum(), 3))             # ずれの大きさの合計
print(round((t ** 2).sum(), 3))              # 得点の二乗和
print(round(((xc - t @ W) ** 2).sum(), 3))   # 誤差


ずれの大きさの合計7.265は、得点の二乗和6.599と、誤差0.666の和です。第1主成分は、誤差がいちばん小さい向きであり、得点がいちばん散らばる向きでもあります。


### TODO②：直交する向きの誤差

第1主成分（45度）に直交（orthogonal）する向き、つまり45＋90＝135度の向きで、誤差を出してください。お手本の`np.deg2rad(30)`を`np.deg2rad(135)`に変えます。


In [ ]:
# TODO: 135度の向きで、得点・近似・誤差を計算し、誤差を表示する

# 解答例②：135度の向きの誤差
theta = np.deg2rad(135)
W = np.array([[np.cos(theta), np.sin(theta)]])
t = xc @ W.T
approx = t @ W
print(round(((xc - approx) ** 2).sum(), 3))


135度の向きの誤差は6.599で、180通りの中でいちばん大きくなります。直交する向きでは、得点の二乗和は0.666しかありません。第1主成分で残しきれなかった散らばりが、この向きに残っています。この向きを、第2主成分と呼びます。2画素なら、2本の向きで、ずれを誤差0で表せます。


---


## 3. 顔の向きはどう見えるか？


2画素では、誤差がいちばん小さい向きを、角度を回して探せました。4,096画素では、向きは4,096個の数の並びになり、角度を回して探すことはできません。同じ考え方の計算を、`PCA`に任せます。


### お手本：4,096画素で主成分分析

- 役割：訓練の300枚に主成分分析をかけ、向きの数と大きさ、平均を確かめる
- 入力：訓練の画素`X_tr`
- 出力：向きの表の大きさと、`PCA`の平均と1節の平均の顔の差

`n_components`を指定しないと、求められるだけの向きを求めます。`random_state=0`は、中の計算で乱数（random number）を使う場合に、毎回同じ結果にする指定です。`mean_`は、`PCA`が引いた平均です。


In [ ]:
pca = PCA(random_state=0).fit(X_tr)
print(pca.components_.shape)
print(np.abs(pca.mean_ - mean_face).max())


向きは300本で、1本が4,096個の数です。300枚から求めるので、向きは枚数までしか作れません。`PCA`の平均は、1節の平均の顔と同じです（差の最大は0）。


### 人の番号は使っていない

`PCA(...).fit(X_tr)`には、人の番号`p_tr`を渡していません。主成分分析は、答えの列を使わずに、画素の表だけから向きを求めます。

答えの列を使う解析を、教師あり学習と呼びました。答えの列を使わず、データそのものの形を調べる解析を、教師なし学習（unsupervised learning）と呼びます。主成分分析は、教師なし学習の1つです。


### お手本：向きを顔として描く

- 役割：最初の5本の向き（4,096個ずつ）を、顔の画像として描く
- 入力：`pca.components_`
- 出力：5枚の画像

向きも4,096個の数なので、`reshape(64, 64)`で画像にできます。顔のデータの向きを顔として描いたものを、固有顔（eigenface）と呼びます。白い画素は向きの値が大きく、黒い画素は小さい（マイナス）ことを表します。


In [ ]:
show_faces(pca.components_[:5],
           ['第1', '第2', '第3', '第4', '第5'])


第1主成分は、顔全体がほぼ同じ色です。顔全体の明るさの違いを表す向きです。第2主成分は、左右で色が分かれています。光が左右どちらから当たっているかの違いです。第3主成分には眼鏡のあたり、第5主成分には口元の違いが出てきます。顔の形の違いは、3本目より先の向きに入っています。

写真どうしの違いが大きい順に、向きが並んでいます。いちばん大きい違いは、顔の形ではなく、撮ったときの光の当たり方でした。


### 寄与率：それぞれの向きが残す散らばり

2節で、得点の二乗和と誤差の和は、ずれの大きさの合計になりました。ずれの大きさの合計のうち、ある向きの得点が受け持つ割合を、その主成分の寄与率（proportion of variance explained）と呼びます。

$$\text{第}k\text{主成分の寄与率} = \frac{\text{第}k\text{主成分の得点の二乗和}}{\text{ずれの大きさの合計}}$$

- 役割：最初の5本の寄与率を出す
- 入力：`pca`
- 出力：5本の寄与率


In [ ]:
print(pca.explained_variance_ratio_[:5].round(3))


第1主成分の寄与率は0.23です。顔の画像のずれのうち、23%を、1本の向きで受け持っています。第2主成分は0.141、第3主成分は0.081と、だんだん小さくなります。


### 確かめる：行列の分解で、同じ向き

`PCA`の中では、特異値分解（singular value decomposition, SVD）という計算で向きを求めています。特異値分解は、ずれの表（行列）を、3つの行列の積に分ける計算です。numpyの`np.linalg.svd`で、同じ向きが出るかを確かめます。

- 役割：ずれの表`X_tr - mean_face`を特異値分解し、3つ目の行列`Vt`の1行目と、`PCA`の第1主成分を比べる
- 入力：訓練の画素と平均の顔
- 出力：`Vt`の大きさ、2つの向きの積の絶対値、`S`から出した寄与率

長さ1の向きどうしの積（`@`）は、同じ向きなら1（逆向きなら−1）になります。`full_matrices=False`は、必要な大きさだけを返す指定です。


In [ ]:
U, S, Vt = np.linalg.svd(X_tr - mean_face, full_matrices=False)
print(Vt.shape)
print(abs(Vt[0] @ pca.components_[0]).round(3))
print((S ** 2 / (S ** 2).sum())[:3].round(3))


`Vt`の行が、`PCA`の向きと同じです（積の絶対値が1）。`S`の二乗の割合が、寄与率と一致します。`PCA`の1行は、この行列の計算をまとめたものです。


---


## 4. 少ない数から顔を戻せるか？


向きが求まりました。顔1枚を、最初の何本かの向きの得点だけで持てば、数が少なくなります。その少ない数から、顔を戻せるでしょうか。


### 記号で書く：縮める・戻す

2節の1本の向きを、$k$本に広げます。$k$本の向き（1本が4,096個）を縦に並べた$k$行4,096列の行列を$\mathbf{W}$、ずれの表（1行が1枚）を$\mathbf{X} - \bar{\mathbf{x}}$と書きます。

- 縮める（得点）：$\mathbf{T} = (\mathbf{X} - \bar{\mathbf{x}})\,\mathbf{W}^\top$　1枚が$k$個の数になる
- 戻す（再構成）：$\hat{\mathbf{X}} = \mathbf{T}\,\mathbf{W} + \bar{\mathbf{x}}$　$k$個の数から4,096画素に戻す

2節の`t = xc @ W.T`と`t @ W`と、同じ形です。少ない数から元の形に戻すことを、再構成（reconstruction）と呼びます。


In [ ]:
# @title 図：行列の大きさで見る（テスト100枚、50本）
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.axis('off'); ax.set_xlim(0, 11); ax.set_ylim(0, 3.2)
def box(x, w, h, c, t):
    ax.add_patch(plt.Rectangle((x, 1.6 - h / 2), w, h, facecolor=c, edgecolor='#333'))
    ax.text(x + w / 2, 1.6 - h / 2 - 0.3, t, ha='center', va='top', fontsize=12)
box(0.2, 3.0, 1.6, '#d8ecfb', 'ずれ 100×4,096')
ax.text(3.6, 1.6, '@', fontsize=18, ha='center', va='center')
box(4.0, 0.4, 2.2, '#fde2e4', '向きの転置\n4,096×50')
ax.text(4.9, 1.6, '=', fontsize=18, ha='center', va='center')
box(5.3, 0.4, 1.6, '#fff1c1', '得点\n100×50')
ax.annotate('', xy=(7.4, 1.6), xytext=(6.0, 1.6), arrowprops=dict(arrowstyle='->', lw=2))
ax.text(6.7, 1.95, '@ 向き ＋ 平均', ha='center', fontsize=12)
box(7.7, 3.0, 1.6, '#e3f4ef', '再構成 100×4,096')
plt.show()


### お手本：テストの顔を50本で縮めて戻す

- 役割：最初の50本の向きで、テストの100枚を得点にし、得点から再構成する。テストの1枚目の写真と再構成を並べる
- 入力：テストの画素`X_te`、向き`pca.components_`、平均の顔
- 出力：得点と再構成の大きさ、2枚の顔

`pca.components_[:50]`は、最初の50行（50本の向き）です。向きと平均は訓練の300枚で決めたものを、そのままテストに使います（第3回の「訓練だけで決める」）。


In [ ]:
W = pca.components_[:50]           # 50本の向き：50行4,096列
T = (X_te - mean_face) @ W.T       # 得点：100行50列
X_hat = T @ W + mean_face          # 再構成：100行4,096列
print(T.shape, X_hat.shape)
show_faces([X_te[0], X_hat[0]], ['写真', '50本で再構成'])


テストの100枚が、1枚50個の数になりました。その50個から戻した顔は、元の写真とよく似ています。

この顔は、主成分を求めた訓練の300枚には入っていません。まだ見ていない顔でも、50本の向きで、おおよそ表せています。


### 確かめる：PCAのtransformも同じ計算

`PCA`の`transform`は、得点を出す部品です。お手本の`T`の先頭と、`transform`の先頭を比べます。


In [ ]:
print(T[0, :3].round(3))
print(pca.transform(X_te)[0, :3].round(3))


同じ数です。`transform`の中身は、ずれ`@`向きの転置でした。第7回で読んだ`StandardScaler`の`transform`と同じく、`fit`で覚えたもの（平均と向き）を使って、新しい行を変換します。


### TODO③：本数を変えて戻す

テストの1枚目`X_te[0]`を、向きの本数`k`を5・20・50・100に変えて再構成し、元の写真と並べて描いてください。`for`の中で、お手本の`50`を`k`に変えます。テストの1枚目だけなので、`X_te[[0]]`（1行の表）で計算します。


In [ ]:
# TODO: k = 5, 20, 50, 100 で X_te[0] を再構成し、写真と並べる

# 解答例③：本数を変えて戻す
faces = [X_te[0]]
for k in [5, 20, 50, 100]:
    W_k = pca.components_[:k]
    T_k = (X_te[[0]] - mean_face) @ W_k.T
    faces.append((T_k @ W_k + mean_face)[0])
show_faces(faces, ['写真', '5本', '20本', '50本', '100本'])


5本では、眼鏡のあたりが分かるだけの、ぼんやりした顔です。20本で、ひげや口元が出てきて、50本・100本と増やすほど、細かいところまで戻ります。数を増やすほど、元の写真に近づきます。

どこまで増やせば十分かを、次の節で数で確かめます。


---


## 5. 何本でどれだけ残るか？


本数を増やすほど、元の写真に近づきました。何本でどれだけ残るかを、2つの数で確かめます。訓練の300枚の散らばりがどれだけ残るか（累積寄与率）と、テストの顔をどれだけ正確に戻せるか（再構成誤差）です。


### お手本：累積寄与率

最初の$k$本の寄与率を足したものを、累積寄与率（cumulative proportion of variance explained）と呼びます。$k$本の向きで、ずれの大きさの合計のうち何割を受け持つかを表します。

- 役割：寄与率を前から順に足していき、1・10・50・100本での累積寄与率を出し、線で描く
- 入力：`pca.explained_variance_ratio_`
- 出力：4つの累積寄与率と、線の図

`cumsum`は、前から順に足していった並びを返します。`cum[9]`は10本目までの和です（位置は0から数えるので、1本ずれます）。


In [ ]:
cum = pca.explained_variance_ratio_.cumsum()
print(cum[[0, 9, 49, 99]].round(3))
plt.plot(range(1, 301), cum)
plt.xlabel('主成分の本数')
plt.ylabel('累積寄与率')
plt.show()


1本で0.23、10本で0.655、50本で0.88、100本で0.945です。はじめの数十本で急に増え、そのあとはゆっくり1に近づきます。300本すべてを使えば、1（訓練の300枚を誤差0で表せる）です。


### TODO④：初めて0.9を超える本数

累積寄与率が初めて0.9以上になる本数を出してください。`cum >= 0.9`はTrue/Falseの並びで、`np.argmax`は、いちばん大きい値（Trueは1、Falseは0）が最初に出る位置を返します。位置は0から数えるので、本数は`+ 1`します。


In [ ]:
# TODO: 累積寄与率が初めて 0.9 以上になる本数を出す

# 解答例④：累積寄与率が初めて0.9以上になる本数
print(np.argmax(cum >= 0.9) + 1)


61本で、累積寄与率が初めて0.9を超えます。4,096個の数のうち、61個で、訓練の顔の散らばりの9割を表せます。


### お手本：テストの顔の再構成誤差

累積寄与率は、向きを求めた訓練の300枚の数です。まだ見ていないテストの顔も、同じように戻せるかを確かめます。

- 役割：本数`k`を変えて、テストの100枚を再構成し、画素1つあたりの誤差の二乗の平均（再構成誤差）を出す。平均の顔だけで戻したとき（0本）の誤差も出す
- 入力：テストの画素、向き、平均の顔
- 出力：本数ごとの再構成誤差と、線の図


In [ ]:
ks = [0, 5, 10, 20, 50, 100, 200]
rec_err = []
for k in ks:
    W_k = pca.components_[:k]
    X_hat_k = (X_te - mean_face) @ W_k.T @ W_k + mean_face
    rec_err.append(((X_te - X_hat_k) ** 2).mean())
print(np.round(rec_err, 4))
plt.plot(ks, rec_err, marker='o')
plt.xlabel('主成分の本数')
plt.ylabel('テストの再構成誤差')
plt.show()


平均の顔だけ（0本）では0.0202、50本で0.0035、100本で0.0026です。テストの顔でも、本数を増やすほど誤差は小さくなります。ただし、50本を超えると、減り方はゆるやかになります。

ここまでで分かったのは、50本ほどで、顔の見た目がかなり残ることです。けれども、顔認証で大事なのは、見た目がどれだけ戻るかではなく、誰の顔かを見分けられるかです。


---


## 6. 見分ける力は残るか？


50本ほどで、顔の見た目はかなり戻りました。では、50個の数で持った顔で、誰の顔かを見分けられるでしょうか。まず、見分ける方法を決めます。


### 1-最近傍法

カメラに写った顔に、いちばん近い登録の顔を探し、その人だと答える方法を、1-最近傍法（1-nearest neighbor）と呼びます。近さは、2つの顔の、画素ごとの差の二乗の和で測ります。

$$d(\mathbf{x}, \mathbf{x}_j) = \sum_{\text{画素}} (\text{差})^2 = \|\mathbf{x} - \mathbf{x}_j\|^2$$

ここでは、訓練の300枚を登録の顔、テストの顔をカメラに写った顔とみなします。


### お手本：1枚で、いちばん近い顔を探す

- 役割：テストの1枚目と、訓練の300枚それぞれの距離を出し、いちばん近い1枚の人の番号を答える
- 入力：訓練の画素と人の番号、テストの1枚目
- 出力：いちばん近い距離、その人の番号、本当の番号

`X_tr - x`は、300行4,096列の表から、4,096個の並び`x`を引く計算です。numpyは、`x`を300行ぶん繰り返して、各行から引きます（ブロードキャストと呼びます）。1節で平均の顔を引いたのも、この形でした。`sum(axis=1)`は、横の方向（行ごと）に足します。


In [ ]:
x = X_te[0]
d = ((X_tr - x) ** 2).sum(axis=1)  # 300枚それぞれとの距離
near = np.argmin(d)
print(round(d[near], 2), p_tr[near], p_te[0])
show_faces([x, X_tr[near]], ['テストの顔', 'いちばん近い顔'])


いちばん近い顔は、番号が同じ人でした。当たりです。


### 部品：KNeighborsClassifier

いちばん近い顔を探して答える部品が、scikit-learnの`KNeighborsClassifier`です。`n_neighbors=1`で、いちばん近い1枚の答えを使います。

- `fit`：登録の顔と人の番号を、そのまま覚える
- `predict`：新しい顔ごとに、覚えた顔のうちいちばん近い1枚の番号を返す
- `score`：当てた割合（正解率）を返す（第6回）


### お手本：画素そのままで交差検証

テストは最後に1回だけ使いたいので、本数の比べ合いは、訓練の300枚の交差検証で行います（第4回）。人ごとの割合をそろえて5つの組に分けます（`StratifiedKFold`、第6回）。

- 役割：4,096画素そのままの1-最近傍法を、5つの組で交差検証し、正解率を出す
- 入力：訓練の画素と人の番号
- 出力：5回分の正解率と、その平均

分類（classification）の部品では、`cross_val_score`は正解率を返します。


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
knn = KNeighborsClassifier(n_neighbors=1)
sc_raw = cross_val_score(knn, X_tr, p_tr, cv=skf)
print(sc_raw.round(3))
print(round(sc_raw.mean(), 3))


4,096画素そのままでは、交差検証の正解率は0.907です。40人から、9割ほどを当てています。これを、縮めたときに比べる相手にします。


### お手本：50本に縮めてから見分ける

- 役割：主成分分析（50本）と1-最近傍法をつないだパイプラインを作り、交差検証で正解率を出す
- 入力：訓練の画素と人の番号
- 出力：5回分の正解率と、その平均

パイプラインに入れたので、交差検証の各回で、`PCA`はその回の訓練の顔だけで向きを求めます（第7回・第8回の「答えを使う手順は中へ」と同じ形です。`PCA`は答えを使いませんが、評価用の顔を見ずに向きを決めるためです）。


In [ ]:
pipe50 = make_pipeline(PCA(n_components=50, random_state=0),
                       KNeighborsClassifier(n_neighbors=1))
sc_50 = cross_val_score(pipe50, X_tr, p_tr, cv=skf)
print(sc_50.round(3))
print(round(sc_50.mean(), 3))


50本に縮めても、交差検証の正解率は0.9です。4,096画素そのままの0.907と、ほとんど変わりません。

累積寄与率は0.88で、散らばりの1割強は捨てています。それでも見分ける力が残るのは、捨てた向きの多くが、人の違いではなく、細かい雑音のような違いだからです。人を見分ける違いは、はじめの数十本の向きに入っています。


### TODO⑤：20本では？

お手本の`n_components=50`を20に変えて、交差検証の正解率の平均を出してください。


In [ ]:
# TODO: 20本に縮めてから見分けるパイプラインで、交差検証の正解率を出す

# 解答例⑤：20本に縮めてから見分ける
pipe20 = make_pipeline(PCA(n_components=20, random_state=0),
                       KNeighborsClassifier(n_neighbors=1))
print(round(cross_val_score(pipe20, X_tr, p_tr, cv=skf).mean(), 3))


20本では、正解率は0.877です。50本の0.9より下がります。少なくしすぎると、見分ける力も落ちていきます。


---


## 7. 何個の数で持てばよいか？


50本なら見分ける力が残り、20本では少し落ちました。最後に、何個の数で持つかを決め、まだ見ていないテストの100枚で1回だけ確かめます。


### 50個を選ぶ

交差検証の正解率は、4,096画素そのままで0.907、50本で0.9、20本で0.877でした。50本は、4,096画素とほぼ同じ力で、数は次のとおり少なくなります。

- 役割：50個が4,096個の何%かを出す
- 入力：なし
- 出力：割合


In [ ]:
print(round(50 / 4096 * 100, 1), '%')


1枚を、元の1.2%の数で持てます。ここでは、50個を選びます。選ぶのは、テストではなく交差検証の結果です（第8回）。


### お手本：テストで1回だけ確かめる

- 役割：4,096画素そのままの1-最近傍法と、50本のパイプラインを、訓練の300枚全員で作り、テストの100枚の正解率を出す
- 入力：訓練とテストの画素と人の番号
- 出力：2つの正解率


In [ ]:
knn.fit(X_tr, p_tr)
pipe50.fit(X_tr, p_tr)
print(knn.score(X_te, p_te))
print(pipe50.score(X_te, p_te))


テストの100枚では、4,096画素そのままも50本も、正解率は0.93です。50本にしても、当たった枚数は変わりません。


### お手本：外れた顔を見る

- 役割：50本のパイプラインで外れたテストの顔を数え、先頭の4枚を描く
- 入力：50本のパイプライン、テストの画素と人の番号
- 出力：外れた枚数と、外れた顔


In [ ]:
pred = pipe50.predict(X_te)
miss = X_te[pred != p_te]
print(len(miss), '枚')
show_faces(miss[:4], ['外れ1', '外れ2', '外れ3', '外れ4'])


外れたのは7枚です。どれも、人の目には普通の正面の写真です。それでも、画素の差の二乗和で測ると、表情や明るさの似た別の人の写真のほうが近くなることがあります。


### 答え

顔の画像1枚（4,096画素）は、主成分分析で50個の数（元の1.2%）にしても、誰の顔かを見分ける力はほとんど落ちません。交差検証の正解率は4,096画素そのままで0.907、50本で0.9でした。テストの100枚では、どちらも0.93でした。50本の累積寄与率は0.88で、捨てた1割強は、見分けるのにあまり役立たない違いでした。

20本まで減らすと、正解率は0.877に下がります。開発チームには、50個を目安に持つことを提案できます。


まだ言えないこと

- ほかの撮り方でも同じか：このデータは、同じ部屋で、顔の位置をそろえて撮った写真。明るさや顔の位置が違うカメラの写真では、主成分の向きそのものが変わりうる
- 40人より多いときも同じか：登録する人が増えると、見分けるのに必要な本数も増えうる
- 登録していない人をどう扱うか：1-最近傍法は、必ず誰かを答える。知らない人を「登録なし」と答える仕組みは、別に要る


---


## 目標に答えられたか


- 今日の目標は「顔の画像1枚を、見分ける力を保ったまま、何個の数で持てばよいか」でした。答えを、本数と正解率で説明できますか？
- 1節で、4,096個の数の並びを顔に戻すには、どうしましたか？ 平均の顔とずれは、何を表しますか？
- 2節で、誤差がいちばん小さい向きが、得点がいちばん散らばる向きでもあるのはなぜですか？
- 3節で、主成分分析が教師なし学習なのはなぜですか？ 第1・第2主成分の固有顔は、何の違いを表していましたか？
- 4節で、縮める・戻すを、行列の積でどう書きましたか？
- 5節の累積寄与率と再構成誤差は、それぞれ何を測っていましたか？
- 6節で、累積寄与率が0.88でも見分ける力が残ったのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：主成分分析は、データを平均からのずれの「誤差が最小になる向き」で近似する。縮めるのも戻すのも行列の積で書け、顔なら4,096個の数を50個にしても、誰の顔かを見分ける力はほとんど落ちない
- 画像は、画素の明るさの並び。`reshape(64, 64)`で画像に戻せる。どの顔も、平均の顔＋ずれ
- 第1主成分は、ずれを1本の向きで近似したときの誤差が最小の向き。三平方の定理から、得点の散らばりが最大の向きでもある
- 主成分分析は答えの列を使わない（教師なし学習）。向きを顔として描いたものが固有顔
- 得点＝ずれ`@`向きの転置、再構成＝得点`@`向き＋平均
- 累積寄与率は訓練の散らばりが何割残るか、再構成誤差はまだ見ていない顔をどれだけ戻せるか。見分ける力は、交差検証の正解率で別に確かめる
- Python：`mean(axis=0)`は縦の方向、`sum(axis=1)`は横の方向。表から1行を引くと、各行から引かれる（ブロードキャスト）


---


## 次回へ


今日は、主成分の向きを顔として描き、第1主成分は顔全体の明るさ、第2主成分は光の当たり方だと読みました。画像なら、向きを絵にして読めます。

次回の問いは、身長や価格のように、単位も意味も違う列が並んだデータでは、主成分の向きや得点を、どう読めばよいか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

6節のパイプラインの主成分の本数を10・30・100に変えて、交差検証の正解率の平均を並べてください。`for`で本数`k`を回し、`PCA(n_components=k, random_state=0)`とします。


In [ ]:
for k in [10, 30, 100]:
    pipe = make_pipeline(PCA(n_components=k, random_state=0),
                         KNeighborsClassifier(n_neighbors=1))
    sc = cross_val_score(pipe, X_tr, p_tr, cv=skf)
    print(k, round(sc.mean(), 3))


<details><summary>詰まったら</summary>

`for k in [10, 30, 100]:`の中で、パイプラインを作り、`cross_val_score(pipe, X_tr, p_tr, cv=skf).mean()`を表示します。
本編の50本は0.9、20本は0.877、4,096画素そのままは0.907でした。

</details>


### 応用②（判断）

応用①の3つ（10・30・100本）と本編の20本・50本のうち、交差検証の正解率が、4,096画素そのまま（0.907）より0.01以上は下がらない、いちばん少ない本数はどれですか。本数を整数で答えてください（例: 25）。


In [ ]:
best_k = None
for k in [10, 20, 30, 50, 100]:
    pipe = make_pipeline(PCA(n_components=k, random_state=0),
                         KNeighborsClassifier(n_neighbors=1))
    v = cross_val_score(pipe, X_tr, p_tr, cv=skf).mean()
    if best_k is None and sc_raw.mean() - v < 0.01:
        best_k = k
print('答え:', best_k)


<details><summary>詰まったら</summary>

4,096画素そのままの正解率から、それぞれの正解率を引き、差が0.01より小さいものの中で、いちばん少ない本数を選びます。テストの正解率では選ばないようにしましょう。

</details>


### 応用③（解釈）

開発チームに向けて、今日の解析で分かったことを3行程度で報告してください。1枚を何個の数で持つとよいか、そのときの見分ける力（正解率）、まだ確かめていないこと、を入れましょう。


（模範例）

顔の画像1枚（4,096画素）は、主成分分析で50個の数（元の1.2%）にしても、誰の顔かを見分ける力はほとんど落ちませんでした（交差検証の正解率0.9、4,096画素そのままで0.907。テストの100枚ではどちらも0.93）。

20本まで減らすと正解率は0.877に下がるので、50個を目安に持つことを提案します。

ただし、同じ部屋で顔の位置をそろえて撮った40人の写真での結果なので、実際のゲートのカメラの写真や、登録していない人の扱いは、別に確かめる必要があります。


---


## 発展（任意）


### 固有顔の向きに、平均の顔を動かす

3節で、第1主成分は顔全体の明るさ、第2主成分は光の当たり方と読みました。平均の顔を、それぞれの向きに沿って動かすと、本当にそう変わるかを確かめます。

- 役割：平均の顔に、第1主成分の向きを、得点の標準偏差の−2倍から＋2倍まで足した5枚を描く
- 入力：平均の顔、`pca`
- 出力：5枚の顔

`pca.explained_variance_`は、各主成分の得点の分散です。`np.sqrt`で標準偏差にします。


In [ ]:
sd = np.sqrt(pca.explained_variance_[0])   # 第1主成分の得点の標準偏差
faces = []
for c in [-2, -1, 0, 1, 2]:
    faces.append(mean_face + c * sd * pca.components_[0])
show_faces(faces, ['−2', '−1', '0', '+1', '+2'])


第1主成分の向きに動かすと、顔全体の明るさが変わります（＋の側では顔が明るく、まわりが暗くなります）。真ん中（0）が平均の顔です。

試すなら、`[0]`を`[1]`に変えて、第2主成分の向きに動かしてみましょう。＋の側では、顔の左半分が明るく、右半分が暗くなります。光が当たる向きの違いです。`[2]`や`[3]`では、どんな違いが見えるでしょうか。


### 第8回の「まとまりの平均」は、主成分とどう違うか

第8回では、TRIM32と一緒に動く527本の遺伝子のまとまりを、標準化した発現量の平均で1本の点数にしました。同じ527本に主成分分析をかけ、第1主成分の得点と、平均の点数を比べます。

- 役割：第8回と同じ手順で、遺伝子を10のまとまりに分け、7番のまとまり（527本）の、平均の点数と第1主成分の得点の相関係数を出す
- 入力：第8回の遺伝子発現のデータ
- 出力：まとまりの大きさ、第1主成分の寄与率、相関係数

第8回のセルを1つにまとめたものです。読み込みと階層的クラスタリングで、少し時間がかかります。


In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
rat = pd.read_csv(f'{DATA_DIR}/rat_eye_full.csv.gz', index_col=0)
g_tr, g_te = train_test_split(rat.drop(columns='TRIM32'),
                              test_size=0.25, random_state=0)
g_tr = g_tr.loc[:, ~g_tr.columns.str.startswith('AFFX')]
g_tr = g_tr.loc[:, g_tr.mean() > g_tr.mean().median()]
var_top = g_tr.var().sort_values(ascending=False).index[:2000]
Z = linkage(g_tr[var_top].T, 'average', metric='correlation')
groups = pd.Series(fcluster(Z, 10, 'maxclust'), index=var_top)
Zs = (g_tr[var_top] - g_tr[var_top].mean()) / g_tr[var_top].std()
g7 = Zs.loc[:, groups == 7]
avg7 = g7.mean(axis=1)                       # 第8回の点数
pca7 = PCA(n_components=1).fit(g7)
s7 = pca7.transform(g7)[:, 0]                # 第1主成分の得点
print(g7.shape)
print(pca7.explained_variance_ratio_.round(3))
print(np.corrcoef(avg7, s7)[0, 1].round(3))


527本のまとまりでは、第1主成分の寄与率は0.394で、平均の点数と第1主成分の得点の相関係数は0.994です。ほとんど同じ点数です。

平均は、527本すべてに同じ重みを付けて足す縮め方です。主成分分析は、誤差がいちばん小さくなるように、遺伝子ごとに重み（向きの値）を決めます。一緒に動く遺伝子のまとまりでは、どの遺伝子もほぼ同じ重みになるので、2つはほぼ一致します。第8回の平均は、第1主成分のいちばん単純な近似だったのです。

試すなら、`pca7.components_[0]`の値を`plt.hist`で描き、527本の重みが、すべて同じ重み（$1/\sqrt{527}$）のまわりに集まるかを見てみましょう。
